# Módulos y lectura de archivos

Hasta ahora, todos los datos se han escrito directamente en el código. En la práctica, la información geográfica se guarda en **archivos**: tablas de estaciones, listas de coordenadas, capas vectoriales o imágenes ráster.

Este cuaderno presenta dos herramientas necesarias para trabajar con ellos:

- **Importar módulos**, para reutilizar código ya escrito. En particular, el módulo `pathlib`, que permite indicar dónde se encuentra un archivo.
- **Abrir archivos** con `open()`, para leer su contenido y escribir resultados.

## Módulos

Un **módulo** es un archivo de Python (extensión `.py`) que contiene funciones, variables y otros elementos listos para reutilizarse. Python incluye una amplia colección de módulos, la **biblioteca estándar** (*standard library*), disponible sin instalar nada más.

Los **paquetes**, presentados en el capítulo de instalación, agrupan varios módulos relacionados. Los paquetes de terceros, como GeoPandas o Rasterio, deben instalarse aparte.

Para usar un módulo hay que **importarlo**, de forma similar a cargar un paquete con `library()` en R.

### Importar un módulo completo

La palabra clave `import` carga un módulo. El módulo `math` contiene funciones y constantes matemáticas. Sus elementos se usan escribiendo el nombre del módulo, un punto y el nombre del elemento:

In [1]:
import math

print(math.pi)
print(math.sqrt(16))

3.141592653589793
4.0


En un cuaderno basta con importar un módulo una vez: queda disponible en todas las celdas que se ejecuten después.

Por ejemplo, la distancia entre dos puntos en coordenadas proyectadas (como UTM, en metros) se calcula con el teorema de Pitágoras:

In [2]:
x1, y1 = 676000, 4613000
x2, y2 = 679500, 4617200

distance_m = math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
print(f"Distance: {distance_m:.1f} m")

Distance: 5467.2 m


La función `help()` también muestra la documentación de los elementos de un módulo:

In [3]:
help(math.sqrt)

Help on built-in function sqrt in module math:

sqrt(x, /)
    Return the square root of x.



### Importar elementos concretos

Con `from ... import ...` se importan solo los elementos indicados. Después se usan directamente por su nombre, sin escribir el del módulo delante:

In [4]:
from math import pi, sqrt

print(pi)
print(sqrt(16))

3.141592653589793
4.0


Ambas formas son correctas. `import math` deja claro de dónde procede cada función; `from math import sqrt` acorta el código cuando se usan pocos elementos con frecuencia.

### Importar con un alias

La palabra clave `as` asigna un nombre alternativo (**alias**) al módulo. Muchas bibliotecas tienen alias acordados por la comunidad, como `import numpy as np` o `import geopandas as gpd`.

In [5]:
import statistics as stats

elevations = [247, 541, 900, 667, 1894]
print(stats.mean(elevations))
print(stats.median(elevations))

849.8
667


### Comprender un error: `ModuleNotFoundError`

Si el nombre del módulo está mal escrito o el módulo no está instalado, Python lanza un `ModuleNotFoundError`. Esta celda falla a propósito:

In [6]:
import maths

ModuleNotFoundError: No module named 'maths'

### Estilo de las importaciones

La [PEP 8](https://peps.python.org/pep-0008/#imports) establece varias pautas sobre cómo y dónde escribir las importaciones.

Cada módulo se importa en una línea distinta:

```python
# Correct
import math
import statistics

# Incorrect
import math, statistics
```

En cambio, varios elementos de un mismo módulo sí pueden importarse en una única línea con `from ... import ...`:

```python
from math import pi, sqrt
```

Las importaciones se escriben siempre **al principio** del archivo, antes del resto del código. Así, cualquier persona que lea el programa identifica de un vistazo los módulos necesarios para ejecutarlo.

Al escribir un programa largo, a veces se añade una importación dentro del bloque en el que se está trabajando para probar una idea. Si finalmente se mantiene, conviene trasladarla al principio del archivo.

Las importaciones siguen además un orden predecible. Se agrupan así, separando cada grupo con una línea en blanco:

1. Módulos de la biblioteca estándar, como `math` o `statistics`.
2. Paquetes de terceros, desarrollados y mantenidos por otras personas u organizaciones. No forman parte de la biblioteca estándar y deben instalarse aparte, como GeoPandas.
3. Módulos propios del proyecto.

```python
import math
import statistics

import geopandas as gpd
```

En este cuaderno las importaciones aparecen a lo largo del texto para presentarlas una a una.

## Rutas de archivos con `pathlib`

### Rutas absolutas y relativas

Una **ruta** (*path*) indica la ubicación de un archivo o una carpeta (también llamada **directorio**):

- Una **ruta absoluta** parte de la raíz del disco, por ejemplo, `C:/courses/intro_python/data/rivers.txt`.
- Una **ruta relativa** parte del **directorio de trabajo** (*working directory*), la carpeta desde la que se ejecuta Python, por ejemplo, `data/rivers.txt`. En Jupyter, el directorio de trabajo es la carpeta que contiene el cuaderno.

Las rutas relativas permiten que el código funcione en otro ordenador, siempre que se mantenga la estructura de carpetas del proyecto.

Escribir rutas como texto (`str`) presenta dos problemas:

- Windows separa las carpetas con `\`, mientras que macOS y Linux usan `/`.
- Dentro de un `str`, `\` inicia caracteres especiales: en `"C:\new_data"`, `\n` se interpreta como un salto de línea.

### El tipo `Path`

El módulo `pathlib` de la biblioteca estándar ofrece `Path`, un tipo de dato que representa rutas y funciona igual en cualquier sistema operativo. Al igual que un `str` o una `list`, un valor de tipo `Path` es un **objeto**: un valor con sus propios métodos. Cómo se crean tipos propios se verá en el cuaderno de clases.

`Path.cwd()` (*current working directory*) devuelve el directorio de trabajo, en este caso la carpeta `notebooks` del repositorio:

In [7]:
from pathlib import Path

working_dir = Path.cwd()
print(working_dir)
print(type(working_dir))

C:\iranzo\doctorado_drive\docencia\master_tig\intro_python\notebooks
<class 'pathlib.WindowsPath'>


En Windows el tipo se muestra como `WindowsPath`; en macOS y Linux, como `PosixPath`. En ambos casos se trabaja con él de la misma manera.

### Construir rutas

El repositorio del curso tiene esta estructura:

```
intro_python/
├── data/
│   ├── rivers.txt
│   └── weather_stations.csv
└── notebooks/
    └── 08_import_open.ipynb
```

Para llegar a `data` desde `notebooks` hay que subir un nivel. Para ello se usa `.parent`, un **atributo**: un valor asociado a un objeto que se consulta con un punto, como los métodos, pero sin paréntesis.

El operador `/` une partes de una ruta con el separador adecuado para cada sistema operativo:

In [8]:
project_dir = Path.cwd().parent
data_dir = project_dir / "data"
rivers_path = data_dir / "rivers.txt"

print(project_dir)
print(data_dir)
print(rivers_path)

C:\iranzo\doctorado_drive\docencia\master_tig\intro_python
C:\iranzo\doctorado_drive\docencia\master_tig\intro_python\data
C:\iranzo\doctorado_drive\docencia\master_tig\intro_python\data\rivers.txt


### Partes de una ruta

Un objeto `Path` también se puede crear a partir de un texto. Sus atributos permiten extraer cada parte de la ruta:

In [9]:
dem_path = Path("data/rasters/dem_zaragoza_25m.tif")

print(dem_path.name)    # File name with extension
print(dem_path.stem)    # File name without extension
print(dem_path.suffix)  # Extension
print(dem_path.parent)  # Containing folder

dem_zaragoza_25m.tif
dem_zaragoza_25m
.tif
data\rasters


Este archivo no existe en el repositorio: un objeto `Path` solo describe una ruta, no comprueba que exista.

El método `.with_suffix()` devuelve una ruta nueva con otra extensión. Es útil, por ejemplo, para localizar los archivos que acompañan a un *shapefile*:

In [10]:
roads_path = Path("data/roads.shp")
projection_path = roads_path.with_suffix(".prj")

print(projection_path)

data\roads.prj


### Comprobar si una ruta existe

Los métodos `.exists()`, `.is_file()` e `.is_dir()` devuelven `True` o `False`:

In [11]:
print(rivers_path.exists())
print(rivers_path.is_file())
print(data_dir.is_dir())
print(dem_path.exists())

True
True
True
False


`dem_path` es una ruta relativa, así que se busca desde el directorio de trabajo (`notebooks/data/rasters/...`) y no existe.

### Listar el contenido de una carpeta

- `.iterdir()` recorre todos los elementos de una carpeta.
- `.glob()` recorre solo los elementos cuyo nombre sigue un patrón. El asterisco (`*`) equivale a «cualquier texto»: `"*.csv"` selecciona todos los archivos con extensión `.csv`.

El orden en que se obtienen los elementos no está garantizado, por lo que se ordenan con `sorted()`:

In [12]:
for path in sorted(data_dir.iterdir()):
    print(path.name)

rivers.txt
weather_stations.csv


In [13]:
for csv_path in sorted(data_dir.glob("*.csv")):
    print(csv_path.name)

weather_stations.csv


En SIG, este patrón permite procesar todas las capas de una carpeta, por ejemplo, con `"*.shp"` o `"*.tif"`.

## Ejercicios

### Partes de una ruta

- Crear un objeto `Path` para la ruta `data/landsat/LC09_L2SP_199031_20240715_B4.TIF`.
- Imprimir el nombre del archivo, el nombre sin extensión y la extensión.
- A partir del nombre sin extensión y con `.split("_")`, imprimir la fecha de adquisición y la banda.

### Inventario de datos

- Construir la ruta a la carpeta `data` del repositorio a partir de `Path.cwd()`.
- Imprimir el nombre de todos los archivos `.txt` que contiene.
- Comprobar si existe en ella un archivo llamado `roads.shp`.

## Leer y escribir archivos con `open()`

### Archivos de texto

Un **archivo de texto** contiene caracteres legibles organizados en líneas, como `.txt`, `.csv`, `.geojson` o `.prj`. Otros formatos, como `.tif` o `.shp`, son **binarios** y se leen con bibliotecas específicas.

La función `open()` abre un archivo y devuelve un objeto con el que leerlo o escribirlo. Recibe:

- La ruta del archivo, como `Path` o como `str`.
- El **modo**: `"r"` para leer (*read*), `"w"` para escribir (*write*) o `"a"` para añadir al final (*append*). Si se omite, se usa `"r"`.
- La **codificación** (*encoding*): la forma en que los caracteres se guardan en el archivo. Conviene indicar siempre `encoding="utf-8"`; de lo contrario, caracteres como `ñ` o las tildes pueden leerse mal según el sistema operativo.

### La instrucción `with`

Un archivo abierto debe **cerrarse** al terminar de usarlo, para liberarlo y asegurar que los cambios se guardan. La instrucción `with` lo cierra automáticamente al salir del bloque con sangría, incluso si se produce un error. La palabra clave `as` asigna el archivo abierto a una variable.

Su estructura es la siguiente (es un esquema, no código ejecutable):

```python
with open(file_path, "r", encoding="utf-8") as file:
    # Instructions that use the open file

# Here the file is already closed
```

### Leer todo el contenido

El método `.read()` devuelve el contenido completo del archivo en un único `str`:

In [14]:
with open(rivers_path, encoding="utf-8") as file:
    content = file.read()

print(content)
print(type(content))

ebro
tajo
duero
guadiana
guadalquivir
júcar
miño
segura

<class 'str'>


La variable `content` sigue disponible después del bloque `with`, aunque el archivo ya esté cerrado.

Los objetos `Path` ofrecen un atajo equivalente, `.read_text()`, que abre, lee y cierra el archivo en una sola instrucción:

In [15]:
print(rivers_path.read_text(encoding="utf-8"))

ebro
tajo
duero
guadiana
guadalquivir
júcar
miño
segura



### Leer línea a línea

Un archivo abierto se puede recorrer con un bucle `for`. En cada iteración se obtiene una línea:

In [16]:
with open(rivers_path, encoding="utf-8") as file:
    for line in file:
        print(line)

ebro

tajo

duero

guadiana

guadalquivir

júcar

miño

segura



Aparecen líneas en blanco intermedias porque cada línea del archivo termina con un salto de línea (`\n`) y `print()` añade otro. El método `.strip()` lo elimina:

In [17]:
rivers = []

with open(rivers_path, encoding="utf-8") as file:
    for line in file:
        rivers.append(line.strip().title())

print(rivers)

['Ebro', 'Tajo', 'Duero', 'Guadiana', 'Guadalquivir', 'Júcar', 'Miño', 'Segura']


### Leer una tabla CSV

Un archivo **CSV** (*comma-separated values*) guarda una tabla como texto: cada línea es una fila y las columnas se separan con comas. La primera línea suele ser la **cabecera**, con los nombres de las columnas:

In [18]:
stations_path = data_dir / "weather_stations.csv"

print(stations_path.read_text(encoding="utf-8"))

name,latitude,longitude,elevation_m
Zaragoza,41.66,-1.01,247
Huesca,42.08,-0.33,541
Teruel,40.35,-1.12,900
Madrid,40.41,-3.68,667
Navacerrada,40.79,-4.01,1894
Logroño,42.45,-2.33,353
Barcelona,41.39,2.20,6
Valencia,39.48,-0.37,11
Sevilla,37.42,-5.88,34
Bilbao,43.30,-2.91,42



Para trabajar con estos datos:

1. `.readlines()` devuelve una lista con todas las líneas del archivo.
2. Con `lines[1:]` se omite la cabecera.
3. `.split(",")` divide cada línea en una lista de valores, que se asignan a varias variables a la vez, como al desempaquetar una tupla.
4. Todo lo que se lee de un archivo de texto es `str`, así que los valores numéricos se convierten con `float()` o `int()`.
5. Cada fila se guarda como un diccionario dentro de una lista.

In [19]:
with open(stations_path, encoding="utf-8") as file:
    lines = file.readlines()

stations = []
for line in lines[1:]:  # Skip the header
    name, latitude, longitude, elevation = line.strip().split(",")
    station = {
        "name": name,
        "latitude": float(latitude),
        "longitude": float(longitude),
        "elevation_m": int(elevation),
    }
    stations.append(station)

print(f"Stations loaded: {len(stations)}")
print(stations[0])

Stations loaded: 10
{'name': 'Zaragoza', 'latitude': 41.66, 'longitude': -1.01, 'elevation_m': 247}


Este procedimiento manual muestra cómo se estructura un CSV, pero falla si algún valor contiene comas. Más adelante se usarán herramientas que simplifican este trabajo, como el módulo `csv` o el paquete pandas.

### Comprender un error: `FileNotFoundError`

Si la ruta no corresponde a ningún archivo, Python lanza un `FileNotFoundError`. Esta celda falla a propósito (el nombre correcto es `weather_stations.csv`):

In [20]:
with open(data_dir / "weather_station.csv", encoding="utf-8") as file:
    content = file.read()

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\iranzo\\doctorado_drive\\docencia\\master_tig\\intro_python\\data\\weather_station.csv'

Ante este error, conviene revisar el nombre, la extensión y el directorio de trabajo. El método `.exists()` ayuda a localizar el problema.

### Escribir un archivo

El modo `"w"` crea el archivo. **Si ya existe, borra su contenido sin avisar.** El método `.write()` escribe un `str` y, a diferencia de `print()`, no añade un salto de línea al final: hay que incluir `\n`.

Los resultados se guardan en una carpeta `output`, separada de los datos originales. El método `.mkdir()` crea la carpeta; `exist_ok=True` evita un error si ya existe.

In [21]:
output_dir = project_dir / "output"
output_dir.mkdir(exist_ok=True)

high_stations_path = output_dir / "high_stations.csv"

with open(high_stations_path, "w", encoding="utf-8") as file:
    file.write("name,elevation_m\n")
    for station in stations:
        if station["elevation_m"] > 500:
            file.write(f"{station['name']},{station['elevation_m']}\n")

print(high_stations_path.read_text(encoding="utf-8"))

name,elevation_m
Huesca,541
Teruel,900
Madrid,667
Navacerrada,1894



### Añadir contenido a un archivo

El modo `"a"` escribe al final del archivo sin borrar lo anterior y lo crea si no existe. Es útil, por ejemplo, para un registro de procesos:

In [22]:
log_path = output_dir / "processing_log.txt"

with open(log_path, "a", encoding="utf-8") as file:
    file.write("Weather stations processed.\n")

print(log_path.read_text(encoding="utf-8"))

Weather stations processed.



Cada vez que se ejecuta la celda se añade una línea nueva. Con el modo `"w"`, el archivo contendría siempre una sola línea.

## Ejercicios

### Lista de ríos

- Leer `rivers.txt` y guardar los nombres en una lista, con la primera letra en mayúscula.
- Imprimir cuántos ríos contiene.
- Escribir en `output/rivers_sorted.txt` los nombres ordenados alfabéticamente, uno por línea.

### Altitud media

- Leer `weather_stations.csv` y calcular la altitud media de las estaciones.
- Imprimir el nombre de las estaciones situadas por encima de la media.

### Estaciones por huso UTM

- Copiar en este cuaderno la función `get_utm_zone()` del cuaderno `07_functions.ipynb`.
- Escribir en `output/stations_utm.csv` una cabecera y una fila por estación con su nombre y su huso UTM.
- Comprobar el resultado con `.read_text()`.

## Resumen

- Un **módulo** es un archivo `.py` con código reutilizable. La **biblioteca estándar** incluye muchos módulos listos para usar.
- `import module` importa un módulo completo; `from module import name`, elementos concretos; `import module as alias`, un módulo con otro nombre.
- Las importaciones se escriben al principio del programa, según la PEP 8.
- `Path`, del módulo `pathlib`, representa rutas de forma independiente del sistema operativo.
- `Path.cwd()` devuelve el directorio de trabajo; `.parent`, la carpeta superior; y el operador `/` une partes de una ruta.
- `.name`, `.stem` y `.suffix` extraen partes de una ruta; `.exists()`, `.is_file()` e `.is_dir()` comprueban si existe; `.iterdir()` y `.glob()` recorren una carpeta.
- `open(path, mode, encoding="utf-8")` abre un archivo en modo lectura (`"r"`), escritura (`"w"`) o adición (`"a"`).
- `with` cierra el archivo automáticamente al terminar el bloque.
- `.read()` lee todo el contenido, un bucle `for` lo recorre línea a línea y `.readlines()` devuelve una lista de líneas. Todo lo leído es `str`.
- `.write()` escribe texto sin añadir saltos de línea.